# Data Understanding

## Skin-Deep Insights: Aspect-Based Sentiment Analysis of Sephora Reviews

**Business Problem:** Many highly rated beauty products (4★+) still experience
hidden dissatisfaction. Star ratings alone do not reveal which specific product
aspects are causing problems.

**Goal of This Notebook:** Build the analytical foundation before any cleaning
or modeling. Answer three questions:
1. What data do we actually have?
2. Is it usable for Aspect-Based Sentiment Analysis?
3. What are the risks and biases in this dataset?

**Approach:**
```
Inspect product table → Inspect reviews table → Assess data quality
→ Document risks → Confirm ABSA viability
```

### Product Table

The product table contains product-level metadata. We inspect it first to
understand what business dimensions are available for segmentation —
brand, price, and category.

**Import Library and load data**

In [1]:
import pandas as pd

In [2]:
df= pd.read_csv("product_info.csv")

**Dataset Structure**

In [3]:
df.shape

(8494, 27)

In [4]:
df.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8494 entries, 0 to 8493
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          8494 non-null   object 
 1   product_name        8494 non-null   object 
 2   brand_id            8494 non-null   int64  
 3   brand_name          8494 non-null   object 
 4   loves_count         8494 non-null   int64  
 5   rating              8216 non-null   float64
 6   reviews             8216 non-null   float64
 7   size                6863 non-null   object 
 8   variation_type      7050 non-null   object 
 9   variation_value     6896 non-null   object 
 10  variation_desc      1250 non-null   object 
 11  ingredients         7549 non-null   object 
 12  price_usd           8494 non-null   float64
 13  value_price_usd     451 non-null    float64
 14  sale_price_usd      270 non-null    float64
 15  limited_edition     8494 non-null   int64  
 16  new   

**Data Type Assessment — Product Table:**

Core business fields (`product_id`, `product_name`, `brand_name`, `price_usd`)
are all non-null and correctly typed. Rating and reviews show 278 missing
values — these are products with no customer feedback yet, expected for a
large catalog. High missingness in pricing variants (`value_price_usd`,
`sale_price_usd`) is expected — these are optional promotional fields not
required for ABSA.

In [6]:
df.describe()

,brand_id,loves_count,rating,reviews,price_usd,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,child_count,child_max_price,child_min_price
count,8494.000000,8.494000e+03,8216.000000,8216.000000,8494.000000,451.000000,270.000000,8494.000000,8494.000000,8494.000000,8494.000000,8494.000000,8494.000000,2754.000000,2754.000000
mean,5422.440546,2.917957e+04,4.194513,448.545521,51.655595,91.168537,20.207889,0.070285,0.071698,0.219096,0.073699,0.279374,1.631622,53.792023,39.665802
std,1709.595957,6.609212e+04,0.516694,1101.982529,53.669234,79.195631,24.327352,0.255642,0.258002,0.413658,0.261296,0.448718,5.379470,58.765894,38.685720
min,1063.000000,0.000000e+00,1.000000,1.000000,3.000000,0.000000,1.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,3.000000
25%,5333.000000,3.758000e+03,3.981725,26.000000,25.000000,45.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22.000000,19.000000
50%,6157.500000,9.880000e+03,4.289350,122.000000,35.000000,67.000000,14.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,32.000000,28.000000
75%,6328.000000,2.684125e+04,4.530525,418.000000,58.000000,108.500000,25.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,59.000000,42.000000
max,8020.000000,1.401068e+06,5.000000,21281.000000,1900.000000,617.000000,320.000000,1.000000,1.000000,1.000000,1.000000,1.000000,105.000000,570.000000,400.000000


**Missing values**

In [7]:
df.isnull().sum()

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                 278
reviews                278
size                  1631
variation_type        1444
variation_value       1598
variation_desc        7244
ingredients            945
price_usd                0
value_price_usd       8043
sale_price_usd        8224
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights            2207
primary_category         0
secondary_category       8
tertiary_category      990
child_count              0
child_max_price       5740
child_min_price       5740
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

**Missing Value Assessment — Product Table:**

Critical fields for analysis (`product_id`, `brand_name`, `price_usd`,
`primary_category`) have zero missing values — product metadata is
structurally sound. High missingness is concentrated in optional fields
(`variation_desc` 85%, `value_price_usd` 95%) which are excluded from
Phase 1 scope.

**Conclusion:** Product metadata is reliable for brand, category, and
pricing analysis.

**Rating Distribution**

In [9]:
df['rating'].value_counts().sort_index()

rating
1.0000     12
1.1905      1
1.2500      1
1.3333      1
1.5000      2
         ... 
4.9818      1
4.9825      1
4.9841      1
4.9937      1
5.0000    256
Name: count, Length: 4394, dtype: int64

In [10]:
df['rating'].value_counts(normalize=True*100)

rating
5.0000    0.031159
4.0000    0.021178
4.5000    0.009981
4.3333    0.008277
3.0000    0.008033
            ...   
4.6954    0.000122
4.2841    0.000122
4.3613    0.000122
4.6878    0.000122
4.6367    0.000122
Name: proportion, Length: 4394, dtype: float64

**Products and brands**

In [11]:
df['product_id'].nunique()

8494

In [12]:
df['brand_name'].nunique()

304

In [13]:
df['product_id'].value_counts().describe()

count    8494.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: count, dtype: float64

### Reviews Table

The reviews table is the core of this project. Every downstream step —
VADER scoring, aspect extraction, demographic segmentation — depends
on the quality of this table. We inspect it thoroughly before proceeding.

**Load Data of one table**

In [14]:
reviews_1 = pd.read_csv("reviews_0-250.csv", on_bad_lines='skip')
print(reviews_1.shape)

/var/folders/_4/sclc4l610q36b3bg5c1xkksh0000gn/T/ipykernel_58322/2445987098.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews_1 = pd.read_csv("reviews_0-250.csv", on_bad_lines='skip')


(602130, 19)


In [15]:
reviews_1.head()

,Unnamed: 0,author_id,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [16]:
reviews_1.columns

Index(['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness',
       'total_feedback_count', 'total_neg_feedback_count',
       'total_pos_feedback_count', 'submission_time', 'review_text',
       'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color',
       'product_id', 'product_name', 'brand_name', 'price_usd'],
      dtype='object')

In [17]:
reviews_1.duplicated().sum()

np.int64(0)

In [18]:
reviews_1.duplicated(subset=['review_text']).sum()

np.int64(93662)

In [19]:
reviews_1.isnull().sum()

Unnamed: 0                       0
author_id                        0
rating                           0
is_recommended              117486
helpfulness                 331832
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                    999
review_title                167011
skin_tone                   106056
eye_color                   138488
skin_type                    74683
hair_color                  141081
product_id                       0
product_name                     0
brand_name                       0
price_usd                        0
dtype: int64

In [20]:
reviews_1['rating'].value_counts().sort_index()

rating
1     32903
2     29253
3     44018
4    106956
5    389000
Name: count, dtype: int64

In [21]:
reviews_1['rating'].value_counts(normalize=True)

rating
5    0.646040
4    0.177629
3    0.073104
1    0.054644
2    0.048583
Name: proportion, dtype: float64

**Rating Distribution — Sample File:**

Already showing strong positive skew — 64.6% are 5★ reviews. This pattern
will persist in the full combined dataset. The implication for sentiment
modeling is important: the dataset is class-imbalanced, and negative signal
will be underrepresented. Standard accuracy metrics will be misleading —
aspect-level complaint rates will need to be interpreted as proportions of
mentions, not absolute counts.

In [22]:
reviews_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 602130 entries, 0 to 602129
Data columns (total 19 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Unnamed: 0                602130 non-null  int64  
 1   author_id                 602130 non-null  object 
 2   rating                    602130 non-null  int64  
 3   is_recommended            484644 non-null  float64
 4   helpfulness               270298 non-null  float64
 5   total_feedback_count      602130 non-null  int64  
 6   total_neg_feedback_count  602130 non-null  int64  
 7   total_pos_feedback_count  602130 non-null  int64  
 8   submission_time           602130 non-null  object 
 9   review_text               601131 non-null  object 
 10  review_title              435119 non-null  object 
 11  skin_tone                 496074 non-null  object 
 12  eye_color                 463642 non-null  object 
 13  skin_type                 527447 non-null  o

**Column Categories — Reviews Table:**

| Category | Columns | ABSA Relevance |
|---|---|---|
| Core NLP | review_text, rating | Critical — cannot proceed without these |
| Demographics | skin_type, skin_tone | Segmentation gold |
| Product | product_id, product_name, brand_name, price_usd | Joining & filtering |
| Time | submission_time | Trend analysis |
| Engagement | helpfulness, total_feedback_count | Advanced signals (Phase 2) |

The presence of skin_type, price_usd, and submission_time in the reviews
table means we do not need to merge with the product table for Phase 1
analysis — everything required is already here.

In [23]:
reviews_1.isnull().sum().sort_values(ascending=False)

helpfulness                 331832
review_title                167011
hair_color                  141081
eye_color                   138488
is_recommended              117486
skin_tone                   106056
skin_type                    74683
review_text                    999
brand_name                       0
product_name                     0
product_id                       0
Unnamed: 0                       0
author_id                        0
submission_time                  0
total_pos_feedback_count         0
total_neg_feedback_count         0
total_feedback_count             0
rating                           0
price_usd                        0
dtype: int64

In [24]:
reviews_1['review_text'].isnull().mean() * 100

np.float64(0.16591101589357782)

In [25]:
reviews_1['skin_type'].isnull().mean() * 100

np.float64(12.403135535515586)

In [26]:
(reviews_1.isnull().mean() * 100).sort_values(ascending=False)

helpfulness                 55.109694
review_title                27.736701
hair_color                  23.430322
eye_color                   22.999684
is_recommended              19.511733
skin_tone                   17.613472
skin_type                   12.403136
review_text                  0.165911
brand_name                   0.000000
product_name                 0.000000
product_id                   0.000000
Unnamed: 0                   0.000000
author_id                    0.000000
submission_time              0.000000
total_pos_feedback_count     0.000000
total_neg_feedback_count     0.000000
total_feedback_count         0.000000
rating                       0.000000
price_usd                    0.000000
dtype: float64

In [27]:
reviews_1['review_length'] = reviews_1['review_text'].str.split().str.len()

In [28]:
reviews_1['review_length'].describe()

count    601131.000000
mean         59.580858
std          43.394992
min           1.000000
25%          31.000000
50%          49.000000
75%          76.000000
max         982.000000
Name: review_length, dtype: float64

**Review Length Assessment:**

Average review length of ~60 words is well above the minimum threshold for
meaningful aspect extraction. Reviews below 5 words (1,163 in this file)
will be removed during cleaning — phrases like "Love it" or "Bad" contain
no extractable aspect information. The maximum of 982 words indicates some
very detailed reviews exist, which will yield multiple aspect mentions per
review.

In [29]:
reviews_1[reviews_1['review_length'] < 5].shape

(1163, 20)

In [30]:
reviews_1['product_id'].nunique()

250

In [31]:
reviews_1['skin_type'].value_counts()

skin_type
combination    292308
dry             99574
normal          69435
oily            66130
Name: count, dtype: int64

In [32]:
reviews_1['skin_type'].value_counts(normalize=True)

skin_type
combination    0.554194
dry            0.188785
normal         0.131644
oily           0.125378
Name: proportion, dtype: float64

In [33]:
reviews_1['review_date'] = pd.to_datetime(reviews_1['submission_time'])

In [34]:
reviews_1['review_date'].min(), reviews_1['review_date'].max()

(Timestamp('2008-08-28 00:00:00'), Timestamp('2023-03-21 00:00:00'))

### Reviews Table

In [35]:
reviews_1 = pd.read_csv("reviews_0-250.csv", on_bad_lines='skip')
reviews_2 = pd.read_csv("reviews_250-500.csv", on_bad_lines='skip')
reviews_3 = pd.read_csv("reviews_500-750.csv", on_bad_lines='skip')
reviews_4 = pd.read_csv("reviews_750-1250.csv", on_bad_lines='skip')
reviews_5 = pd.read_csv("reviews_1250-end.csv", on_bad_lines='skip')

reviews = pd.concat([reviews_1, reviews_2, reviews_3, reviews_4, reviews_5], ignore_index=True)
print(reviews.shape)

/var/folders/_4/sclc4l610q36b3bg5c1xkksh0000gn/T/ipykernel_58322/1342024628.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews_1 = pd.read_csv("reviews_0-250.csv", on_bad_lines='skip')
/var/folders/_4/sclc4l610q36b3bg5c1xkksh0000gn/T/ipykernel_58322/1342024628.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews_4 = pd.read_csv("reviews_750-1250.csv", on_bad_lines='skip')


(1094411, 19)


/var/folders/_4/sclc4l610q36b3bg5c1xkksh0000gn/T/ipykernel_58322/1342024628.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews_5 = pd.read_csv("reviews_1250-end.csv", on_bad_lines='skip')


**Missing Values**

In [36]:
reviews.isnull().sum()

Unnamed: 0                       0
author_id                        0
rating                           0
is_recommended              167988
helpfulness                 561592
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
submission_time                  0
review_text                   1444
review_title                310654
skin_tone                   170539
eye_color                   209628
skin_type                   111557
hair_color                  226768
product_id                       0
product_name                     0
brand_name                       0
price_usd                        0
dtype: int64

In [37]:
reviews.duplicated().sum()

np.int64(0)

In [38]:
reviews.duplicated(subset=['review_text']).sum()

np.int64(124991)

In [39]:
(reviews.isnull().mean() * 100).sort_values(ascending=False)

helpfulness                 51.314543
review_title                28.385497
hair_color                  20.720552
eye_color                   19.154413
skin_tone                   15.582720
is_recommended              15.349626
skin_type                   10.193337
review_text                  0.131943
brand_name                   0.000000
product_name                 0.000000
product_id                   0.000000
Unnamed: 0                   0.000000
author_id                    0.000000
submission_time              0.000000
total_pos_feedback_count     0.000000
total_neg_feedback_count     0.000000
total_feedback_count         0.000000
rating                       0.000000
price_usd                    0.000000
dtype: float64

**Data type**

In [40]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1094411 entries, 0 to 1094410
Data columns (total 19 columns):
 #   Column                    Non-Null Count    Dtype  
---  ------                    --------------    -----  
 0   Unnamed: 0                1094411 non-null  int64  
 1   author_id                 1094411 non-null  object 
 2   rating                    1094411 non-null  int64  
 3   is_recommended            926423 non-null   float64
 4   helpfulness               532819 non-null   float64
 5   total_feedback_count      1094411 non-null  int64  
 6   total_neg_feedback_count  1094411 non-null  int64  
 7   total_pos_feedback_count  1094411 non-null  int64  
 8   submission_time           1094411 non-null  object 
 9   review_text               1092967 non-null  object 
 10  review_title              783757 non-null   object 
 11  skin_tone                 923872 non-null   object 
 12  eye_color                 884783 non-null   object 
 13  skin_type                 9

**Rating Distribution**

In [41]:
reviews['rating'].value_counts().sort_index()

rating
1     61223
2     53032
3     81816
4    199389
5    698951
Name: count, dtype: int64

In [42]:
reviews['rating'].value_counts(normalize=True)

rating
5    0.638655
4    0.182188
3    0.074758
1    0.055942
2    0.048457
Name: proportion, dtype: float64

**Review Length Analysis**

In [43]:
reviews['review_text'].isnull().mean() * 100

np.float64(0.13194311826178648)

In [44]:
reviews['review_length'] = reviews['review_text'].str.split().str.len()

In [45]:
reviews['review_length'].describe()

count    1.092967e+06
mean     6.053948e+01
std      4.348206e+01
min      1.000000e+00
25%      3.200000e+01
50%      5.000000e+01
75%      7.600000e+01
max      1.216000e+03
Name: review_length, dtype: float64

In [46]:
reviews[reviews['review_length'] < 5].shape

(1693, 20)

In [47]:
reviews['product_id'].nunique()

2351

**Skin type Analysis**

In [48]:
reviews['skin_type'].isnull().mean() * 100

np.float64(10.193336872527778)

In [49]:
reviews['skin_type'].value_counts()

skin_type
combination    544513
dry            185937
normal         131910
oily           120494
Name: count, dtype: int64

In [50]:
reviews['skin_type'].value_counts(normalize=True)

skin_type
combination    0.554012
dry            0.189181
normal         0.134211
oily           0.122596
Name: proportion, dtype: float64

**Time Dimension**

In [51]:
reviews['review_date'] = pd.to_datetime(reviews['submission_time'])

In [52]:
reviews['review_date'].min(), reviews['review_date'].max()

(Timestamp('2008-08-28 00:00:00'), Timestamp('2023-03-21 00:00:00'))

In [53]:
high_rated = reviews[reviews['rating'] >= 4]

In [54]:
high_rated.shape

(898340, 21)

**Core Hypothesis Setup:**

898,340 reviews — 82% of the dataset — are rated 4★ or higher. This is
the pool within which we will search for hidden dissatisfaction in Stage 7.

If even 4% of these contain negative aspect-level sentiment in their text,
that represents over 35,000 complaints hidden behind positive star ratings.
The business case for ABSA is already clear from this number alone.

**Price Distribution**

In [55]:
df['price_usd'].describe()

count    8494.000000
mean       51.655595
std        53.669234
min         3.000000
25%        25.000000
50%        35.000000
75%        58.000000
max      1900.000000
Name: price_usd, dtype: float64

In [56]:
reviews['price_usd'].describe()

count    1.094411e+06
mean     4.900838e+01
std      4.004338e+01
min      3.000000e+00
25%      2.500000e+01
50%      3.900000e+01
75%      6.200000e+01
max      1.900000e+03
Name: price_usd, dtype: float64

In [57]:
reviews.to_csv("../sephora/reviews.csv", index=False)

## Data Understanding Summary

### 1. Dataset Structure

Two relational tables linked via `product_id`:

| Table | Rows | Columns | Purpose |
|---|---|---|---|
| product_info.csv | 8,494 | 27 | Product metadata |
| reviews (combined) | 1,094,411 | 19 | Customer feedback |

### 2. Product Table

- 8,494 unique products, 304 unique brands
- Price range: $3 – $1,900 (mean: $51.66)
- Core fields fully complete — reliable for analysis
- Optional pricing fields excluded from scope

### 3. Reviews Table

- 1,094,411 reviews across 2,351 products (28% of catalog)
- Coverage: August 2008 – March 2023 (15 years)
- Average review length: ~60 words
- Missing review_text: 0.13% — negligible
- Missing skin_type: 10.2% — documented limitation

**Rating Distribution:**

| Rating | Count | % |
|---|---|---|
| 5★ | 698,951 | 63.9% |
| 4★ | 199,389 | 18.2% |
| 3★ | 81,816 | 7.5% |
| 2★ | 53,032 | 4.8% |
| 1★ | 61,223 | 5.6% |

82% of reviews are 4★ or 5★ — strong positive skew confirmed.

### 4. Dataset Risks & Biases

| Risk | Detail | Management |
|---|---|---|
| Rating skew | 82% positive | Use aspect-level rates, not counts |
| Skin type imbalance | Combination = 55% | Normalize by segment size in analysis |
| Duplicate review text | 124,991 repeated | Surgical deduplication in Stage 5 |
| Short reviews | 1,693 under 5 words | Filter in Stage 5 |

### 5. ABSA Viability Verdict

- Sufficient volume for statistical analysis (1M+ reviews)
- Demographic segmentation available (skin_type)
- Price metadata available for tier analysis
- Rich review text averaging 60 words per review
- Enough negative signal (18% rated 1★–3★) for contrast

**This dataset is viable for Aspect-Based Sentiment Analysis.**
Proceeding to Stage 4 — Data Extraction.
```